In [ ]:
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

output_dir = os.path.join(project_root, "output")

from qc_engine.parser import load_metadata, load_qc_table
from qc_engine.column_namer import rename_analyte_columns
from qc_engine.classifier import classify_qc_table
from qc_engine.report_metadata import extract_all_identifiers

# ------------------------------------------------------------
# User Inputs (only update these)
# ------------------------------------------------------------
path = "G:/R_and_D/Projects/PrinceGeorge/Carp/2025/Geochemistry/A25-15564Final.xlsx"   # <-- Update file here
lab_name = "Actlabs"                                      # <-- Update lab name here
meta_key = "Report Number"                                # <-- Update metadata key here
date_key = "Report Date"                                  # <-- Update date key here
qc_sheet = "QC"                                           # <-- Update Excel sheet name here

# QC interpretation settings
method_code = "AR-ICPMS"                                     # <-- Update assay method code here. See below for the list of options 
matrix_type = "vegetation"                                      # <-- Update sample type here. See below for a list of options

# QC tolerance settings

crm_low_tol = 80.0                                        # CRM lower recovery tolerance (%), default = 80
crm_high_tol = 120.0                                      # CRM upper recovery tolerance (%), default = 120

duplicate_rpd_tol = 30.0                                  # Duplicate RPD tolerance (%), default = 30

blank_tol_factor = 3.0                                    # Blank exceedance factor (e.g., 3× DL), default = 3

# BDL substitution settings
bdl_sub_rule = "half"                                     # options: "half", "sqrt2", "zero", "dl", or a numeric value



# QC interpretation settings options

valid_method_codes = [
    "AR-ICPMS",
    "AR-ICPOES",
    "4-Acid",
    "FA-AAS",
    "FA-ICP",
    "PhotonAssay",
    "INAA"
]

valid_matrix_types = [
    "rock",
    "soil",
    "till",
    "vegetation",
    "talus",
    "stream sediment",
    "lake sediment"
]

print("Valid method_code options:", valid_method_codes)
print("Valid matrix_type options:", valid_matrix_types)


# ------------------------------------------------------------
# Load metadata + QC table
# ------------------------------------------------------------
metadata = load_metadata(path)      # returns dict
df_qc = load_qc_table(path)

# ------------------------------------------------------------
# Extract identifiers
# ------------------------------------------------------------
file_name, metadata_name, report_date, names_match = extract_all_identifiers(
    path,
    metadata,
    meta_key=meta_key,
    date_key=date_key
)

print("File Name:", file_name)
print("Metadata Name:", metadata_name)
print("Report Date:", report_date)
print("Names Match:", names_match)

# ------------------------------------------------------------
# Continue QC processing
# ------------------------------------------------------------
df_qc = rename_analyte_columns(df_qc, metadata)
df_qc = classify_qc_table(df_qc)

df_qc.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
from qc_engine.crm_wide import compute_crm_recovery

crm_results = compute_crm_recovery(
    df_qc,
    metadata,
    low_tol=crm_low_tol,
    high_tol=crm_high_tol
)

crm_results.head()

In [ ]:
from qc_engine.parser import load_metadata, load_qc_table
from qc_engine.column_namer import rename_analyte_columns
from qc_engine.classifier import classify_qc_table
from qc_engine.duplicate_wide import compute_duplicate_rpd

metadata = load_metadata(path)
df_qc = load_qc_table(path)

df_qc = rename_analyte_columns(df_qc, metadata)
df_qc = classify_qc_table(df_qc)

dup_results = compute_duplicate_rpd(
    df_qc,
    metadata,
    rpd_tolerance=duplicate_rpd_tol
)

dup_results.head()

In [ ]:
from qc_engine.blank_wide import compute_blank_qc

blank_results = compute_blank_qc(
    df_qc,
    metadata,
    tolerance_factor=blank_tol_factor,
    bdl_rule=bdl_sub_rule
)

blank_results.head()

In [ ]:
from qc_engine.qc_summary import build_qc_summary

summary = build_qc_summary(crm_results, dup_results, blank_results)
summary.head()

In [ ]:
# CRM summary for plotting
crm_summary = summary[["Analyte", "CRM_Recovery", "CRM_FinalStatus"]]

# Duplicate summary for plotting
dup_summary = summary[["Analyte", "Duplicate_RPD", "Duplicate_Status"]]

# Blank summary for plotting (use raw blank_results ONLY)
blank_plot_data = blank_results.rename(columns={"Status": "Blank_Status"})


In [ ]:
from qc_engine.qc_plots import plot_crm_recovery, plot_duplicate_rpd, plot_blank_levels

# Build certificate order from metadata
certificate_order = [metadata[i]["analyte"] for i in sorted(metadata.keys())]

# Display plots for visual inspection
fig_crm = plot_crm_recovery(crm_summary, certificate_order)
fig_dup = plot_duplicate_rpd(dup_summary, certificate_order)
fig_blank = plot_blank_levels(blank_plot_data, certificate_order)

In [ ]:
from qc_engine.report_export import save_qc_plot

save_qc_plot(
    fig=fig_crm,
    lab_name=lab_name,
    metadata_name=metadata_name,
    report_date=report_date,
    plot_type="CRM Recovery",
    outdir=output_dir
)

save_qc_plot(
    fig=fig_dup,
    lab_name=lab_name,
    metadata_name=metadata_name,
    report_date=report_date,
    plot_type="Duplicate RPD",
    outdir=output_dir
)

save_qc_plot(
    fig=fig_blank,
    lab_name=lab_name,
    metadata_name=metadata_name,
    report_date=report_date,
    plot_type="Blank Levels",
    outdir=output_dir
)

In [ ]:
import importlib
import qc_engine.interpretation as interp
importlib.reload(interp)

from qc_engine.interpretation import generate_qc_interpretation

In [ ]:
qc_text = generate_qc_interpretation(
    crm_results=crm_results,
    dup_results=dup_results,
    blank_results=blank_results,
    df_qc=df_qc,
    method_code=method_code,
    lab_name=lab_name,
    file_name=file_name,
    metadata_name=metadata_name,
    report_date=report_date,
    matrix_type=matrix_type,
    crm_low_tol=crm_low_tol,
    crm_high_tol=crm_high_tol,
    duplicate_rpd_tol=duplicate_rpd_tol,
    blank_tol_factor=blank_tol_factor,
    bdl_sub_rule=bdl_sub_rule
)


print(qc_text)